# Lisca crop

Edit the **Config** cell, then run the cells below.

Define the assay and align in Studio first. This notebook reads `assay.json` and the ROI boxes in `bbox/`. It crops the ND2 or CZI saved on Metadata, and only the positions that were aligned. It does not ask for sample names or a position list.

A TIFF folder is cropped in Studio. Continue in `analyze.ipynb` after that crop.


## Config

Set the workspace. The image source and the positions come from the assay Studio saved.


In [ ]:
from pathlib import Path

# Folder that holds assay.json and bbox/ from Studio. Not the ND2 or CZI file.
WORKSPACE = Path(r"Z:\projects\LNPbinder\Experiments\20260731\Auswertung")


## Setup


In [ ]:
from lisca.core.assay_positions import load_assay_positions
from lisca.core.bbox import discover_bbox_positions
from lisca.services import crop

workspace = WORKSPACE.expanduser().resolve()
if not workspace.is_dir():
    raise FileNotFoundError(f"Workspace not found: {workspace}")

assay = load_assay_positions(workspace)
if assay.source_type == "folder":
    raise ValueError(
        "This workspace source is a TIFF folder. Crop it in Studio, then continue "
        "in analyze.ipynb. This notebook crops the ND2 or CZI saved on Metadata."
    )
source = assay.source.expanduser().resolve()
if not source.is_file():
    raise FileNotFoundError(f"Source not found: {source}")

aligned = set(discover_bbox_positions(workspace))
positions = [pos for pos in assay.file_positions if pos in aligned]
missing = [pos for pos in assay.file_positions if pos not in aligned]
if not positions:
    raise ValueError(
        "No saved ROI boxes for this assay. Align in Studio and save each position."
    )

print(f"Workspace: {workspace}")
print(f"Source: {source}")
print(f"Samples: {', '.join(assay.samples)}")
print(f"Positions to crop ({len(positions)}): {positions}")
if missing:
    print(f"In the assay but not aligned, skipped: {missing}")


## Crop


In [ ]:
result = crop.run_crop(
    workspace=workspace,
    source=source,
    positions=positions,
    on_progress=print,
)
for written in result.written:
    print(crop.format_written_crop_message(written))
if result.skipped_missing_bbox:
    print(f"Skipped missing bbox CSVs: {result.skipped_missing_bbox}")

print("Crop finished. Continue in analyze.ipynb.")
